In [1]:
# Install the official Google GenAI SDK and Pydantic
!pip install google-genai pydantic --quiet
print("✅ Installation complete! Please restart your runtime if prompted.")

✅ Installation complete! Please restart your runtime if prompted.


In [2]:
import os
from google.colab import userdata
from openai import OpenAI

# Set your API key safely (or use Colab Secrets / Environment Variables)
# Fetch API key directly from Colab secrets
OPENROUTER_API_KEY = userdata.get('Openrouter')

# Initialize the OpenAI client configured for OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL = "poolside/laguna-m.1" # OpenRouter model name

print(f"✅ Client initialized successfully using model: {MODEL}")

✅ Client initialized successfully using model: poolside/laguna-m.1


In [3]:
from pydantic import BaseModel, Field, field_validator, ValidationError
import json

# 1. Define the schema with explicit types and descriptive fields
class Candidate(BaseModel):
    full_name: str = Field(description="Person's full name")
    years_experience: int = Field(description="Total years of work experience")
    primary_skill: str = Field(description="Their single strongest technical skill")
    email: str = Field(description="Contact email address")
    notice_period_days: int | None = Field(
        default=None,
        description="Notice period in days if mentioned, otherwise leave as None"
    )

    # 2. Add custom validation rules
    @field_validator("years_experience")
    @classmethod
    def years_must_be_sane(cls, v: int) -> int:
        if v < 0 or v > 60:
            raise ValueError("years_experience must be a realistic number between 0 and 60")
        return v

# 3. Create a production-grade wrapper with retry logic for OpenAI client
def extract_candidate(text: str, max_retries: int = 2) -> Candidate | None:
    for attempt in range(1, max_retries + 1):
        print(f"🔄 Attempt {attempt} to extract structured data...")
        try:
            response = client.chat.completions.create(
                model=MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": f"Extract information as a JSON object strictly following this Pydantic schema: {json.dumps(Candidate.model_json_schema())}"},
                    {"role": "user", "content": text}
                ]
            )
            # Parse the JSON response and validate with Pydantic
            json_output = json.loads(response.choices[0].message.content)
            return Candidate(**json_output)

        except json.JSONDecodeError as e:
            print(f"⚠️ JSON decoding failed: {e}")
        except ValidationError as e:
            print(f"⚠️ Pydantic validation failed: {e}")
        except Exception as e:
            print(f"⚠️ API or parsing exception encountered: {e}")

    print("❌ Critical: Extraction failed completely after max retries.")
    return None

# 4. Test the implementation with messy input text
messy_text = """
Hey, I'm Priya Sharma. Been coding for about 6 years now, mostly Python
and some data science stuff. You can reach me at priya.codes@gmail.com
whenever. Cheers!
"""

print("--- Running Extraction ---")
candidate_profile = extract_candidate(messy_text)

if candidate_profile:
    print("\n🎉 Success! Clean, Object Data Retreived:")
    print(f"Name       : {candidate_profile.full_name}")
    print(f"Experience : {candidate_profile.years_experience} years")
    print(f"Skill       : {candidate_profile.primary_skill}")
    print(f"Email      : {candidate_profile.email}")
    print(f"Notice-Prd : {candidate_profile.notice_period_days} days")

--- Running Extraction ---
🔄 Attempt 1 to extract structured data...

🎉 Success! Clean, Object Data Retreived:
Name       : Priya Sharma
Experience : 6 years
Skill       : Python
Email      : priya.codes@gmail.com
Notice-Prd : None days


In [4]:
import json

# 1. Define explicit local tools with strong type hints and clear instructions
def get_weather(city: str) -> str:
    """Get the current weather description for a given city.

    Args:
        city: The name of the city, e.g., "Delhi", "London".
    """
    fake_db = {
        "delhi": "34°C, sunny",
        "london": "12°C, rainy",
        "tokyo": "19°C, cloudy",
    }
    return fake_db.get(city.lower(), "No data available for that city")

def multiply(a: float, b: float) -> float:
    """Multiply two numerical values together accurately.

    Args:
        a: The first number.
        b: The second number.
    """
    return a * b

def convert_currency(amount: float, from_cur: str, to_cur: str) -> str:
    """Convert an amount of money from one currency to another using fixed exchange rates.

    Args:
        amount: How much money to convert.
        from_cur: The 3-letter source currency code, e.g., "USD".
        to_cur: The 3-letter target currency code, e.g., "INR".
    """
    rates = {"USD_INR": 83.0, "INR_USD": 1 / 83.0, "EUR_INR": 90.0}
    key = f"{from_cur.upper()}_{to_cur.upper()}"
    if key not in rates:
        return "Exchange rate conversion path not available"
    return f"{amount * rates[key]:.2f} {to_cur.upper()}"

# Helper to convert Python functions to OpenAI tool schema format
def to_openai_tool_schema(func):
    schema = {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": func.__doc__,
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
    import inspect
    sig = inspect.signature(func)
    for name, param in sig.parameters.items():
        if param.kind == inspect.Parameter.POSITIONAL_OR_KEYWORD:
            param_type = "string" # Default, can be refined based on type hints
            if param.annotation is inspect.Parameter.empty:
                 # If no type hint, try to infer from default value or keep string
                 if param.default is not inspect.Parameter.empty and isinstance(param.default, (int, float)):
                     param_type = "number"
                 elif param.default is not inspect.Parameter.empty and isinstance(param.default, bool):
                     param_type = "boolean"
                 else:
                     param_type = "string"
            elif param.annotation == float or param.annotation == int:
                param_type = "number"
            elif param.annotation == str:
                param_type = "string"

            schema["function"]["parameters"]["properties"][name] = {"type": param_type, "description": f"Argument {name}"}
            if param.default is inspect.Parameter.empty:
                schema["function"]["parameters"]["required"].append(name)
    return schema

# 2. Package tools into an accessible list in OpenAI format
tool_suite_openai = [
    to_openai_tool_schema(get_weather),
    to_openai_tool_schema(multiply),
    to_openai_tool_schema(convert_currency)
]

# 3. Test queries to show autonomous routing across different domains
test_queries = [
    "What's the weather like in Tokyo?",
    "Calculate 847 times 219 for me.",
    "Convert 100 US dollars to Indian rupees.",
    "Convert it to rupees for me." # Vague query to test clarification
]

print("--- Automated Multi-Tool Routing ---")
for query in test_queries:
    print(f"\nUser Query: '{query}'")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are an expert helper. If a user request lacks crucial information "
                "required to execute a specific tool accurately, ask a polite clarifying "
                "question instead of hallucinating parameters."
            )},
            {"role": "user", "content": query}
        ],
        tools=tool_suite_openai,
        tool_choice="auto" # Let the model decide whether to call a tool or respond directly
    )

    response_message = response.choices[0].message

    # Check if the model wanted to call a tool
    if response_message.tool_calls:
        print("Tool Call(s) Detected:")
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            print(f"  Function: {function_name}")
            print(f"  Args: {function_args}")

            # Execute the tool (this part would typically be in a separate handler)
            available_functions = {"get_weather": get_weather, "multiply": multiply, "convert_currency": convert_currency}
            function_to_call = available_functions[function_name]
            function_response = function_to_call(**function_args)
            print(f"  Tool Output: {function_response}")

            # Send the tool output back to the model
            second_response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are an expert helper."},
                    {"role": "user", "content": query},
                    response_message,
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": str(function_response),
                    },
                ],
            )
            print(f"Gemini Response: {second_response.choices[0].message.content}")
    else:
        print(f"Gemini Response: {response_message.content}")

--- Automated Multi-Tool Routing ---

User Query: 'What's the weather like in Tokyo?'
Tool Call(s) Detected:
  Function: get_weather
  Args: {'city': 'Tokyo'}
  Tool Output: 19°C, cloudy
Gemini Response: 
The weather in Tokyo is currently 19°C with cloudy conditions.


User Query: 'Calculate 847 times 219 for me.'
Tool Call(s) Detected:
  Function: multiply
  Args: {'a': 847, 'b': 219}
  Tool Output: 185493
Gemini Response: 
847 multiplied by 219 is **185,493**.


User Query: 'Convert 100 US dollars to Indian rupees.'
Tool Call(s) Detected:
  Function: convert_currency
  Args: {'amount': 100, 'from_cur': 'USD', 'to_cur': 'INR'}
  Tool Output: 8300.00 INR
Gemini Response: 
100 US dollars is equivalent to approximately **8300.00 Indian rupees (INR)** based on the current exchange rate.


User Query: 'Convert it to rupees for me.'
Gemini Response: 
I'd be happy to help you convert to rupees! However, I need a bit more information:

- What amount would you like to convert?
- What currency 

In [5]:
from pydantic import BaseModel, Field
import json

# Define target business data structure
class Invoice(BaseModel):
    invoice_number: str = Field(description="The unique identifier or ID of the invoice document")
    vendor: str = Field(description="The name of the company or entity issuing the invoice")
    total_amount: float = Field(description="The final total payable value as a float number")
    due_date: str = Field(description="The absolute deadline payment date transformed into YYYY-MM-DD format")

# Unstructured real-world operational input
incoming_email = """
Hi team, please process payment for invoice INV-2026-0091 from
BrightWare Solutions. The total comes to 45,600 rupees and it's
due by the 15th of August 2026. Thanks!
"""

print("--- Running Invoice Extraction ---")
try:
    response = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": f"Extract invoice information as a JSON object strictly following this Pydantic schema: {json.dumps(Invoice.model_json_schema())}"},
            {"role": "user", "content": incoming_email}
        ]
    )

    # Parse output into system object
    json_output = json.loads(response.choices[0].message.content)
    extracted_invoice: Invoice = Invoice(**json_output)

    print("\nParsed Data Object:")
    print(extracted_invoice)
    print("\nIndividual Attribute Access:")
    print(f"Vendor Code   : {extracted_invoice.vendor}")
    print(f"Total Due     : {extracted_invoice.total_amount}")
    print(f"Due Sequence  : {extracted_invoice.due_date}")
except json.JSONDecodeError as e:
    print(f"⚠️ JSON decoding failed: {e}")
except ValidationError as e:
    print(f"⚠️ Pydantic validation failed: {e}")
except Exception as e:
    print(f"⚠️ API or parsing exception encountered: {e}")

--- Running Invoice Extraction ---

Parsed Data Object:
invoice_number='INV-2026-0091' vendor='BrightWare Solutions' total_amount=45600.0 due_date='2026-08-15'

Individual Attribute Access:
Vendor Code   : BrightWare Solutions
Total Due     : 45600.0
Due Sequence  : 2026-08-15


In [6]:
from pydantic import BaseModel, Field, field_validator, ValidationError
import json

# 1. Define the schema with explicit types and descriptive fields
class Candidate(BaseModel):
    full_name: str = Field(description="Person's full name")
    years_experience: int = Field(description="Total years of work experience")
    primary_skill: str = Field(description="Their single strongest technical skill")
    email: str = Field(description="Contact email address")
    notice_period_days: int | None = Field(
        default=None,
        description="Notice period in days if mentioned, otherwise leave as None"
    )

    # 2. Add custom validation rules
    @field_validator("years_experience")
    @classmethod
    def years_must_be_sane(cls, v: int) -> int:
        if v < 0 or v > 60:
            raise ValueError("years_experience must be a realistic number between 0 and 60")
        return v

# 3. Create a production-grade wrapper with retry logic for OpenAI client
def extract_candidate(text: str, max_retries: int = 2) -> Candidate | None:
    for attempt in range(1, max_retries + 1):
        print(f"🔄 Attempt {attempt} to extract structured data...")
        try:
            response = client.chat.completions.create(
                model=MODEL,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": f"Extract information as a JSON object strictly following this Pydantic schema: {json.dumps(Candidate.model_json_schema())}"},
                    {"role": "user", "content": text}
                ]
            )
            # Parse the JSON response and validate with Pydantic
            json_output = json.loads(response.choices[0].message.content)
            return Candidate(**json_output)

        except json.JSONDecodeError as e:
            print(f"⚠️ JSON decoding failed: {e}")
        except ValidationError as e:
            print(f"⚠️ Pydantic validation failed: {e}")
        except Exception as e:
            print(f"⚠️ API or parsing exception encountered: {e}")

    print("❌ Critical: Extraction failed completely after max retries.")
    return None

# 4. Test the implementation with messy input text
messy_text = """
Hey, I'm Priya Sharma. Been coding for about 6 years now, mostly Python
and some data science stuff. You can reach me at priya.codes@gmail.com
whenever. Cheers!
"""

print("--- Running Extraction ---")
candidate_profile = extract_candidate(messy_text)

if candidate_profile:
    print("\n🎉 Success! Clean, Object Data Retreived:")
    print(f"Name       : {candidate_profile.full_name}")
    print(f"Experience : {candidate_profile.years_experience} years")
    print(f"Skill       : {candidate_profile.primary_skill}")
    print(f"Email      : {candidate_profile.email}")
    print(f"Notice-Prd : {candidate_profile.notice_period_days} days")

--- Running Extraction ---
🔄 Attempt 1 to extract structured data...

🎉 Success! Clean, Object Data Retreived:
Name       : Priya Sharma
Experience : 6 years
Skill       : Python
Email      : priya.codes@gmail.com
Notice-Prd : None days


In [7]:
import json

# 1. Define explicit local tools with strong type hints and clear instructions
def get_weather(city: str) -> str:
    """Get the current weather description for a given city.

    Args:
        city: The name of the city, e.g., "Delhi", "London".
    """
    fake_db = {
        "delhi": "34°C, sunny",
        "london": "12°C, rainy",
        "tokyo": "19°C, cloudy",
    }
    return fake_db.get(city.lower(), "No data available for that city")

def multiply(a: float, b: float) -> float:
    """Multiply two numerical values together accurately.

    Args:
        a: The first number.
        b: The second number.
    """
    return a * b

def convert_currency(amount: float, from_cur: str, to_cur: str) -> str:
    """Convert an amount of money from one currency to another using fixed exchange rates.

    Args:
        amount: How much money to convert.
        from_cur: The 3-letter source currency code, e.g., "USD".
        to_cur: The 3-letter target currency code, e.g., "INR".
    """
    rates = {"USD_INR": 83.0, "INR_USD": 1 / 83.0, "EUR_INR": 90.0}
    key = f"{from_cur.upper()}_{to_cur.upper()}"
    if key not in rates:
        return "Exchange rate conversion path not available"
    return f"{amount * rates[key]:.2f} {to_cur.upper()}"

# Helper to convert Python functions to OpenAI tool schema format
def to_openai_tool_schema(func):
    schema = {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": func.__doc__,
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
    import inspect
    sig = inspect.signature(func)
    for name, param in sig.parameters.items():
        if param.kind == inspect.Parameter.POSITIONAL_OR_KEYWORD:
            param_type = "string" # Default, can be refined based on type hints
            if param.annotation is inspect.Parameter.empty:
                 # If no type hint, try to infer from default value or keep string
                 if param.default is not inspect.Parameter.empty and isinstance(param.default, (int, float)):
                     param_type = "number"
                 elif param.default is not inspect.Parameter.empty and isinstance(param.default, bool):
                     param_type = "boolean"
                 else:
                     param_type = "string"
            elif param.annotation == float or param.annotation == int:
                param_type = "number"
            elif param.annotation == str:
                param_type = "string"

            schema["function"]["parameters"]["properties"][name] = {"type": param_type, "description": f"Argument {name}"}
            if param.default is inspect.Parameter.empty:
                schema["function"]["parameters"]["required"].append(name)
    return schema

# 2. Package tools into an accessible list in OpenAI format
tool_suite_openai = [
    to_openai_tool_schema(get_weather),
    to_openai_tool_schema(multiply),
    to_openai_tool_schema(convert_currency)
]

# 3. Test queries to show autonomous routing across different domains
test_queries = [
    "What's the weather like in Tokyo?",
    "Calculate 847 times 219 for me.",
    "Convert 100 US dollars to Indian rupees.",
    "Convert it to rupees for me." # Vague query to test clarification
]

print("--- Automated Multi-Tool Routing ---")
for query in test_queries:
    print(f"\nUser Query: '{query}'")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are an expert helper. If a user request lacks crucial information "
                "required to execute a specific tool accurately, ask a polite clarifying "
                "question instead of hallucinating parameters."
            )},
            {"role": "user", "content": query}
        ],
        tools=tool_suite_openai,
        tool_choice="auto" # Let the model decide whether to call a tool or respond directly
    )

    response_message = response.choices[0].message

    # Check if the model wanted to call a tool
    if response_message.tool_calls:
        print("Tool Call(s) Detected:")
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            print(f"  Function: {function_name}")
            print(f"  Args: {function_args}")

            # Execute the tool (this part would typically be in a separate handler)
            available_functions = {"get_weather": get_weather, "multiply": multiply, "convert_currency": convert_currency}
            function_to_call = available_functions[function_name]
            function_response = function_to_call(**function_args)
            print(f"  Tool Output: {function_response}")

            # Send the tool output back to the model
            second_response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are an expert helper."},
                    {"role": "user", "content": query},
                    response_message,
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": str(function_response),
                    },
                ],
            )
            print(f"Gemini Response: {second_response.choices[0].message.content}")
    else:
        print(f"Gemini Response: {response_message.content}")

--- Automated Multi-Tool Routing ---

User Query: 'What's the weather like in Tokyo?'
Tool Call(s) Detected:
  Function: get_weather
  Args: {'city': 'Tokyo'}
  Tool Output: 19°C, cloudy
Gemini Response: 
Currently in Tokyo, it's **19°C and cloudy**.


User Query: 'Calculate 847 times 219 for me.'
Tool Call(s) Detected:
  Function: multiply
  Args: {'a': 847, 'b': 219}
  Tool Output: 185493
Gemini Response: 
The result of 847 times 219 is **185,493**.


User Query: 'Convert 100 US dollars to Indian rupees.'
Tool Call(s) Detected:
  Function: convert_currency
  Args: {'amount': 100, 'from_cur': 'USD', 'to_cur': 'INR'}
  Tool Output: 8300.00 INR
Gemini Response: 
**100 US Dollars (USD) = ₹8,300.00 Indian Rupees (INR)**

*Exchange rate used: 1 USD = 83 INR*


User Query: 'Convert it to rupees for me.'
Gemini Response: 
Sure! I can help convert an amount to Indian Rupees (INR) for you. Could you please let me know:

- The amount you’d like to convert  
- The currency it’s currently in (e.g

In [8]:
from pydantic import BaseModel, Field
import json

# Define target business data structure
class Invoice(BaseModel):
    invoice_number: str = Field(description="The unique identifier or ID of the invoice document")
    vendor: str = Field(description="The name of the company or entity issuing the invoice")
    total_amount: float = Field(description="The final total payable value as a float number")
    due_date: str = Field(description="The absolute deadline payment date transformed into YYYY-MM-DD format")

# Unstructured real-world operational input
incoming_email = """
Hi team, please process payment for invoice INV-2026-0091 from
BrightWare Solutions. The total comes to 45,600 rupees and it's
due by the 15th of August 2026. Thanks!
"""

print("--- Running Invoice Extraction ---")
try:
    response = client.chat.completions.create(
        model=MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": f"Extract invoice information as a JSON object strictly following this Pydantic schema: {json.dumps(Invoice.model_json_schema())}"},
            {"role": "user", "content": incoming_email}
        ]
    )

    # Parse output into system object
    json_output = json.loads(response.choices[0].message.content)
    extracted_invoice: Invoice = Invoice(**json_output)

    print("\nParsed Data Object:")
    print(extracted_invoice)
    print("\nIndividual Attribute Access:")
    print(f"Vendor Code   : {extracted_invoice.vendor}")
    print(f"Total Due     : {extracted_invoice.total_amount}")
    print(f"Due Sequence  : {extracted_invoice.due_date}")
except json.JSONDecodeError as e:
    print(f"⚠️ JSON decoding failed: {e}")
except ValidationError as e:
    print(f"⚠️ Pydantic validation failed: {e}")
except Exception as e:
    print(f"⚠️ API or parsing exception encountered: {e}")

--- Running Invoice Extraction ---

Parsed Data Object:
invoice_number='INV-2026-0091' vendor='BrightWare Solutions' total_amount=45600.0 due_date='2026-08-15'

Individual Attribute Access:
Vendor Code   : BrightWare Solutions
Total Due     : 45600.0
Due Sequence  : 2026-08-15


In [9]:
students_data = [
    {
        "id": "S001",
        "name": "Sydney Sweeni",
        "age": 18,
        "grade": "A",
        "subjects": ["Math", "Physics", "Chemistry"],
        "gpa": 3.9
    },
    {
        "id": "S002",
        "name": "Bob Johnson",
        "age": 17,
        "grade": "B",
        "subjects": ["Math", "Biology", "English"],
        "gpa": 3.2
    },
    {
        "id": "S003",
        "name": "Charlie Brown",
        "age": 18,
        "grade": "A",
        "subjects": ["History", "Literature", "Art"],
        "gpa": 4.0
    },
    {
        "id": "S004",
        "name": "Diana Miller",
        "age": 19,
        "grade": "C",
        "subjects": ["Physics", "Computer Science"],
        "gpa": 2.8
    }
]

print("Student data created successfully!")

Student data created successfully!


Now that we have some student data, let's ask for different details about these students.

In [10]:
# 1. List all student names
student_names = [student['name'] for student in students_data]
print(f"All student names: {student_names}")

# 2. Find students who have a grade 'A'
grade_a_students = [student['name'] for student in students_data if student['grade'] == 'A']
print(f"Students with grade 'A': {grade_a_students}")

# 3. Get the average age of students
total_age = sum(student['age'] for student in students_data)
average_age = total_age / len(students_data)
print(f"Average age of students: {average_age:.2f}")

# 4. List subjects for a specific student (e.g., Alice Smith)
alice_subjects = next((student['subjects'] for student in students_data if student['name'] == 'Alice Smith'), "Student not found")
print(f"Subjects for Alice Smith: {alice_subjects}")

# 5. Find students taking 'Physics'
physics_students = [student['name'] for student in students_data if 'Physics' in student['subjects']]
print(f"Students taking Physics: {physics_students}")

All student names: ['Sydney Sweeni', 'Bob Johnson', 'Charlie Brown', 'Diana Miller']
Students with grade 'A': ['Sydney Sweeni', 'Charlie Brown']
Average age of students: 18.00
Subjects for Alice Smith: Student not found
Students taking Physics: ['Sydney Sweeni', 'Diana Miller']
